# DIscarded Experiment 6: N-Grams

**Course Outcome:** CO4
**BT Level:** BT4

## Aim
Generate unigram, bigram, and trigram sequences from text and analyse their frequencies for language modelling.

## Objectives
- Understand n-grams as contiguous token sequences.
- Generate and count unigram, bigram, and trigram sequences using Python and NLTK.
- Compare the context available at each order and the resulting data sparsity.

## Theory
An **n-gram** is a contiguous sequence of *n* tokens. A unigram contains one token, a bigram contains two adjacent tokens, and a trigram contains three. Higher-order n-grams capture more local context, but they are typically less frequent; this is called **data sparsity**.

For a count-based model, a bigram probability can be estimated as $P(w_i\mid w_{i-1}) = C(w_{i-1}, w_i) / C(w_{i-1})$. In practical language models, sentence-boundary symbols and smoothing are used to handle beginnings, endings, and unseen sequences.

## Imports and corpus
If NLTK is not installed, run `%pip install nltk` once in a notebook cell. `wordpunct_tokenize` is used because it is supplied by NLTK and does not need a separate downloaded tokenizer model. The small corpus deliberately repeats a few phrases so that frequency differences are visible.

In [1]:
from collections import Counter
from nltk import ngrams
from nltk.tokenize import wordpunct_tokenize
import re

text = '''Natural language processing helps computers understand human language.
Natural language processing uses language data to identify patterns.
Language models learn useful patterns from text data.
A language model assigns probabilities to word sequences.
Natural language data can improve a language model.'''

# Split sentences without requiring an NLTK data download.
sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]

def tokenize(sentence):
    """Lowercase and retain word-like tokens only."""
    return [token.lower() for token in wordpunct_tokenize(sentence) if token.isalpha()]

tokenized_sentences = [tokenize(sentence) for sentence in sentences]
tokens = [token for sentence in tokenized_sentences for token in sentence]

print('Number of sentences:', len(tokenized_sentences))
print('Number of tokens:', len(tokens))
print('Normalized tokens:', tokens)

Number of sentences: 5
Number of tokens: 41
Normalized tokens: ['natural', 'language', 'processing', 'helps', 'computers', 'understand', 'human', 'language', 'natural', 'language', 'processing', 'uses', 'language', 'data', 'to', 'identify', 'patterns', 'language', 'models', 'learn', 'useful', 'patterns', 'from', 'text', 'data', 'a', 'language', 'model', 'assigns', 'probabilities', 'to', 'word', 'sequences', 'natural', 'language', 'data', 'can', 'improve', 'a', 'language', 'model']


## Generate n-grams and frequency distributions
N-grams are generated sentence by sentence so no sequence incorrectly crosses a sentence boundary.

In [2]:
def corpus_ngrams(tokenized_sentences, n):
    return [gram
            for sentence in tokenized_sentences
            for gram in ngrams(sentence, n)]

unigrams = corpus_ngrams(tokenized_sentences, 1)
bigrams = corpus_ngrams(tokenized_sentences, 2)
trigrams = corpus_ngrams(tokenized_sentences, 3)

unigram_counts = Counter(unigrams)
bigram_counts = Counter(bigrams)
trigram_counts = Counter(trigrams)

print('First 12 unigrams:', unigrams[:12])
print('First 12 bigrams: ', bigrams[:12])
print('First 12 trigrams:', trigrams[:12])

First 12 unigrams: [('natural',), ('language',), ('processing',), ('helps',), ('computers',), ('understand',), ('human',), ('language',), ('natural',), ('language',), ('processing',), ('uses',)]
First 12 bigrams:  [('natural', 'language'), ('language', 'processing'), ('processing', 'helps'), ('helps', 'computers'), ('computers', 'understand'), ('understand', 'human'), ('human', 'language'), ('natural', 'language'), ('language', 'processing'), ('processing', 'uses'), ('uses', 'language'), ('language', 'data')]
First 12 trigrams: [('natural', 'language', 'processing'), ('language', 'processing', 'helps'), ('processing', 'helps', 'computers'), ('helps', 'computers', 'understand'), ('computers', 'understand', 'human'), ('understand', 'human', 'language'), ('natural', 'language', 'processing'), ('language', 'processing', 'uses'), ('processing', 'uses', 'language'), ('uses', 'language', 'data'), ('language', 'data', 'to'), ('data', 'to', 'identify')]


In [3]:
def print_most_common(counter, title, n=10):
    print(f'\n{title}')
    print(f"{'N-gram':<40} Frequency")
    print('-' * 52)
    for gram, count in counter.most_common(n):
        print(f"{' '.join(gram):<40} {count}")

print_most_common(unigram_counts, 'Most frequent unigrams')
print_most_common(bigram_counts, 'Most frequent bigrams')
print_most_common(trigram_counts, 'Most frequent trigrams')


Most frequent unigrams
N-gram                                   Frequency
----------------------------------------------------
language                                 8
natural                                  3
data                                     3
processing                               2
to                                       2
patterns                                 2
a                                        2
model                                    2
helps                                    1
computers                                1

Most frequent bigrams
N-gram                                   Frequency
----------------------------------------------------
natural language                         3
language processing                      2
language data                            2
a language                               2
language model                           2
processing helps                         1
helps computers                          1
computers unde

## Frequency and sparsity comparison
The table reports how many sequences were observed, how many were unique, and the proportion of unique sequences. A larger unique proportion means counts are spread across more distinct sequences, indicating greater sparsity.

In [4]:
ngram_counters = {
    'Unigram': unigram_counts,
    'Bigram': bigram_counts,
    'Trigram': trigram_counts
}

print(f"{'Order':<10} {'Observed':>10} {'Unique':>10} {'Unique proportion':>20} {'Max frequency':>16}")
print('-' * 72)
for name, counter in ngram_counters.items():
    observed = sum(counter.values())
    unique = len(counter)
    print(f"{name:<10} {observed:>10} {unique:>10} {unique / observed:>20.3f} {max(counter.values()):>16}")

Order        Observed     Unique    Unique proportion    Max frequency
------------------------------------------------------------------------
Unigram            41         25                0.610                8
Bigram             36         30                0.833                3
Trigram            31         29                0.935                2


## Sentence-boundary markers and a sample probability
Boundary symbols make sentence starts and ends explicit. For example, the model can represent the chance that `natural` begins a sentence.

In [5]:
# Add one start and one end marker for a bigram model.
boundary_bigrams = [gram
                   for sentence in tokenized_sentences
                   for gram in ngrams(['<s>'] + sentence + ['</s>'], 2)]
boundary_bigram_counts = Counter(boundary_bigrams)
boundary_context_counts = Counter(gram[0] for gram in boundary_bigrams)

target = ('<s>', 'natural')
probability = boundary_bigram_counts[target] / boundary_context_counts[target[0]]
print(f"Count{target} = {boundary_bigram_counts[target]}")
print(f"Count('<s>') = {boundary_context_counts['<s>']}")
print(f"P(natural | <s>) = {probability:.3f}")

Count('<s>', 'natural') = 3
Count('<s>') = 5
P(natural | <s>) = 0.600


## Result and interpretation
Repeated words and expressions, such as **natural language**, have the highest frequency. As the order increases from unigrams to trigrams, each sequence contains more context, but more of the observed sequences are unique and individual counts tend to decrease. This demonstrates the context--sparsity trade-off in n-gram language models.

## Self-assessment answers
1. An n-gram is a contiguous sequence of *n* tokens (words or characters).
2. Unigrams contain one token; bigrams contain two adjacent tokens; trigrams contain three adjacent tokens.
3. Longer n-grams include more preceding tokens, so they describe a more specific local context.
4. Data sparsity is the problem that many possible longer sequences have very small counts or are not observed.
5. The Markov assumption approximates a word's probability using only a limited number of preceding words.
6. Boundary symbols let a model learn which words start or end sentences.
7. An n-gram probability is estimated by dividing its count by the count of its $(n-1)$-gram context.
8. Character n-grams are used in spelling correction and language identification (also useful for plagiarism detection).
9. A zero probability makes any sentence containing that unseen n-gram receive zero probability; smoothing addresses this.
10. Increasing n increases the vocabulary of possible sequences, so memory use and computation grow substantially.